# 02 — Tracking experiments

Multi-object tracking on apple orchard video using **YOLO11 + ByteTrack**.

> **Prerequisite:** `best.pt` from `01_training_experiments.ipynb` must be available
> (downloaded or uploaded to this session).

---

# Section 0 - Install & imports

In [ ]:
# Install dependencies
# Re-run safe — pip skips already-installed packages.
import subprocess, sys

pkgs = [
    "ultralytics>=8.3",   # includes ByteTrack via model.track()
    "supervision>=0.21",  # sv.ByteTracker, sv.LineZone, sv.Annotator helpers
    "opencv-python-headless"
]
subprocess.run([sys.executable, "-m", "pip", "install", *pkgs, "--quiet"])

In [ ]:
# Core imports
import os, shutil, json, math, time
from pathlib  import Path
from datetime import datetime

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from IPython.display import display, Video, Image as IPImage

import torch
from ultralytics import YOLO
import supervision as sv

---

# Section 1 — Configuration

| Parameter | Typical Colab value | Notes |
|-----------|--------------------|----|
| `weights_path` | path to `best.pt` | from Phase 1 |
| `video_path` | path to test video | `.mp4` from the Kaggle dataset |
| `conf` | `0.35` | detection confidence threshold |
| `iou` | `0.45` | NMS IoU threshold |
| `imgsz` | `640` | must match training size |
| `count_line_y` | `0.5` | fractional Y position of the counting line (0-1) |


In [ ]:
CFG = dict(
    # ── model ─────────────────────────────────────────────────────────────
    weights_path = "/content/best.pt",   # upload or copy from drive

    # ── video ─────────────────────────────────────────────────────────────
    # Put the path to any orchard video here. The Kaggle dataset ships with
    # .mp4 files inside raw/videos/  — use one of those for a quick smoke test.
    video_path   = "/content/project_fruit_detection/data/raw/video_test_01.mp4",

    # ── inference ─────────────────────────────────────────────────────────
    conf         = 0.35,
    iou          = 0.45,
    imgsz        = 640,

    # ── ByteTrack ─────────────────────────────────────────────────────────
    # These are supervision's ByteTracker parameters (sensible defaults).
    track_thresh       = 0.35,  # min detection confidence to initialise a track
    track_buffer       = 30,    # frames to keep a lost track alive
    match_thresh       = 0.8,   # IoU threshold for track-to-detection matching

    # ── counting ──────────────────────────────────────────────────────────
    count_line_y = 0.5,   # horizontal line at 50 % of frame height

    # ── output ────────────────────────────────────────────────────────────
    save_video   = True,
    output_dir   = "/content/working/tracking"
)

print("Configuration loaded:")
for k, v in CFG.items():
    print(f"  {k:<22s} {v}")

---

# Section 2 — Environment setup

In [ ]:
# Detect platform & mount drive / authenticate Kaggle
import sys

if "google.colab" in sys.modules or os.path.exists("/content"):
    PLATFORM = "Colab"
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        print("Google Drive mounted.")
    except Exception as e:
        print(f"Drive mount skipped: {e}")
elif os.path.exists("/kaggle"):
    PLATFORM = "Kaggle"
else:
    PLATFORM = "Local"

print(f"Platform: {PLATFORM}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Create output directories
OUTPUT_DIR = Path(CFG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# (Colab only) Download dataset from Kaggle if not already present

KAGGLE_DATASET = "matarauj/project-fruit-detection"
RAW_DIR = Path("/content/project_fruit_detection")

if PLATFORM == "Colab" and not RAW_DIR.exists():
    from google.colab import files
    # Upload kaggle.json first if not already done
    if not Path("/root/.kaggle/kaggle.json").exists():
        print("Upload your kaggle.json ↓")
        uploaded = files.upload()
        Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
        shutil.move(list(uploaded.keys())[0], "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    os.system(f"kaggle datasets download -d {KAGGLE_DATASET} -p /content --unzip -q")
    print(f"Dataset extracted to {RAW_DIR}")
else:
    print(f"Dataset directory exists or not on Colab — skipping download.")

---

# Section 3 — Load detection model

In [ ]:
# Cell 3  Load best.pt from Phase 1
weights = Path(CFG["weights_path"])
if not weights.exists():
    raise FileNotFoundError(
        f"Weights not found at {weights}.\n"
        "Upload best.pt (from the Phase 1 export) to this session,\n"
        "or update CFG['weights_path'] to point at the correct location."
    )

model = YOLO(str(weights))
print(f"Model loaded: {weights.name}")
print(f"  Task  : {model.task}")
print(f"  Classes: {model.names}")

---
# Section 4 — Run ByteTrack on a sample video

`model.track()` wraps detection + ByteTrack in a single call.
We iterate frame-by-frame so we can:
1. Apply custom counting logic (Section 6).
2. Render annotations with `supervision`.
3. Write the annotated frames to an output `.mp4`.

In [ ]:
# Probe video metadata
video_path = Path(CFG["video_path"])
if not video_path.exists():
    raise FileNotFoundError(
        f"Video not found at {video_path}.\n"
        "Update CFG['video_path'] or upload a video file."
    )

cap = cv2.VideoCapture(str(video_path))
FRAME_W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
FRAME_H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS      = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f"Video   : {video_path.name}")
print(f"Size    : {FRAME_W} × {FRAME_H}  @  {FPS:.1f} fps")
print(f"Frames  : {N_FRAMES}  ({N_FRAMES/FPS:.1f} s)")

In [ ]:
# Initialise ByteTracker and annotators
byte_tracker = sv.ByteTracker(
    track_thresh  = CFG["track_thresh"],
    track_buffer  = CFG["track_buffer"],
    match_thresh  = CFG["match_thresh"]
)

# Counting line — horizontal, at fractional Y position
LINE_Y = int(FRAME_H * CFG["count_line_y"])
count_line = sv.LineZone(
    start=sv.Point(0,       LINE_Y),
    end  =sv.Point(FRAME_W, LINE_Y),
)
line_annotator     = sv.LineZoneAnnotator(thickness=2, text_thickness=2, text_scale=0.8)
box_annotator      = sv.BoxAnnotator(thickness=2)
label_annotator    = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
trace_annotator    = sv.TraceAnnotator(thickness=2, trace_length=40)

print(f"ByteTracker ready.")
print(f"Counting line Y: {LINE_Y} px  ({CFG['count_line_y']*100:.0f}% of frame height)")

In [ ]:
# Frame-by-frame tracking loop
# ─────────────────────────────────────────────────────────────────────────────
# This is the main processing loop.  For each frame we:
#   1. Run YOLO11 detection (no re-initialisation of the model — fast).
#   2. Feed detections into ByteTracker → get tracked detections with IDs.
#   3. Update the line-crossing counter.
#   4. Render annotations onto the frame.
#   5. Write to the output video.
# ─────────────────────────────────────────────────────────────────────────────

run_ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = OUTPUT_DIR / f"tracked_{video_path.stem}_{run_ts}.mp4"

cap    = cv2.VideoCapture(str(video_path))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(out_path), fourcc, FPS, (FRAME_W, FRAME_H)) if CFG["save_video"] else None

frame_idx      = 0
total_crossed  = 0       # apples that crossed the line (in direction A→B)
track_ids_seen = set()   # all unique track IDs ever assigned
per_frame_log  = []      # list of dicts for later analysis

t0 = time.time()

while True:
    ok, frame = cap.read()
    if not ok:
        break

    # ── 1. Detect ────────────────────────────────────────────────────────────
    results = model.predict(
        frame,
        conf    = CFG["conf"],
        iou     = CFG["iou"],
        imgsz   = CFG["imgsz"],
        verbose = False
    )[0]

    # Convert YOLO results → supervision Detections
    detections = sv.Detections.from_ultralytics(results)

    # ── 2. Track ─────────────────────────────────────────────────────────────
    detections = byte_tracker.update_with_detections(detections)

    # ── 3. Count (line crossing) ──────────────────────────────────────────────
    count_line.trigger(detections=detections)
    total_crossed  = count_line.in_count + count_line.out_count
    track_ids_seen |= set(detections.tracker_id.tolist()) if detections.tracker_id is not None else set()

    # ── 4. Annotate ───────────────────────────────────────────────────────────
    labels = [
        f"#{tid}  {conf:.2f}"
        for tid, conf in zip(
            detections.tracker_id if detections.tracker_id is not None else [],
            detections.confidence if detections.confidence is not None else []
        )
    ]

    annotated = frame.copy()
    annotated = trace_annotator.annotate(scene=annotated, detections=detections)
    annotated = box_annotator.annotate(scene=annotated, detections=detections)
    annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)
    annotated = line_annotator.annotate(frame=annotated, line_counter=count_line)

    # HUD overlay
    cv2.putText(annotated, f"Frame: {frame_idx:05d}",
                (10, 30),  cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    cv2.putText(annotated, f"Tracked IDs (total): {len(track_ids_seen)}",
                (10, 60),  cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    cv2.putText(annotated, f"Line crossings: {total_crossed}",
                (10, 90),  cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0),   2)

    # ── 5. Write ──────────────────────────────────────────────────────────────
    if writer:
        writer.write(annotated)

    per_frame_log.append({
        "frame"     : frame_idx,
        "n_det"     : len(detections),
        "n_tracks"  : len(detections.tracker_id) if detections.tracker_id is not None else 0,
        "crossed"   : total_crossed
    })

    frame_idx += 1

cap.release()
if writer:
    writer.release()

elapsed = time.time() - t0
print(f"Processed {frame_idx} frames in {elapsed:.1f} s  ({frame_idx/elapsed:.1f} fps)")
print(f"Unique track IDs   : {len(track_ids_seen)}")
print(f"Total line crossings: {total_crossed}  (in: {count_line.in_count}, out: {count_line.out_count})")
if CFG["save_video"]:
    print(f"Output video saved : {out_path}")

---

# Section 5 — Visualise tracked output

In [ ]:
# Show per-frame detection & tracking counts
df = pd.DataFrame(per_frame_log)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(df["frame"], df["n_det"],    label="Detections",  alpha=0.8)
axes[0].plot(df["frame"], df["n_tracks"], label="Active tracks", alpha=0.8)
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].set_title("Detections vs active tracks per frame")

axes[1].plot(df["frame"], df["crossed"], color="green")
axes[1].set_ylabel("Cumulative crossings")
axes[1].set_xlabel("Frame")
axes[1].set_title("Cumulative line crossings over time")

plt.tight_layout()
plt.show()

In [ ]:
# Sample annotated frames from the output video
# Extracts 8 evenly-spaced frames and displays them in a grid.

if CFG["save_video"] and out_path.exists():
    cap_out  = cv2.VideoCapture(str(out_path))
    n_out    = int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT))
    indices  = np.linspace(0, n_out - 1, 8, dtype=int)

    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, idx in zip(axes.flat, indices):
        cap_out.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frm = cap_out.read()
        if ok:
            ax.imshow(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
            ax.set_title(f"Frame {idx}", fontsize=8)
        ax.axis("off")
    cap_out.release()
    plt.suptitle("Sample annotated frames", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No output video to preview (save_video=False or file missing).")

In [ ]:
# (Colab only) Inline video playback
if PLATFORM == "Colab" and CFG["save_video"] and out_path.exists():
    display(Video(str(out_path), embed=True, width=800))

---

# Section 6 — Counting logic

The `sv.LineZone` counter used in Section 4 counts every crossing of the
horizontal line.  For a **harvest-count** use-case (one apple = one unique
fruit passing through the frame) we need **unique track IDs**, not line
crossings — because the same apple could cross the line multiple times if it
oscillates.

In [ ]:
# Two counting strategies

# Strategy A — line crossings (already computed above)
crossing_count = count_line.in_count  # downward crossings (A→B)

# Strategy B — unique track IDs ever seen
unique_track_count = len(track_ids_seen)

# Strategy C — unique IDs that crossed the line (intersection)
# supervision stores crossing history on count_line.tracker_state
# We can inspect which tracker IDs are recorded as "in" crossings.
try:
    crossed_ids = {tid for tid, state in count_line.tracker_state.items() if state}
    unique_crossing_count = len(crossed_ids)
except AttributeError:
    unique_crossing_count = None   # older supervision versions may not expose this

print("─" * 45)
print("Counting summary")
print("─" * 45)
print(f"  A) Line crossings (downward)  : {crossing_count}")
print(f"  B) Unique track IDs (total)   : {unique_track_count}")
if unique_crossing_count is not None:
    print(f"  C) Unique IDs crossing line   : {unique_crossing_count}")
print()
print("For harvest counting, Strategy C (or B) is more reliable.")
print("Strategy A over-counts if an apple oscillates over the line.")

---

# Section 7 — Evaluation on an annotated clip (optional)

If you have a short clip where you know the **ground-truth fruit count**,
fill in `GT_COUNT` below.  The cell computes a simple counting error.

For tracking-quality metrics (MOTA, IDF1, HOTA) you would need per-frame
ground-truth bounding boxes with consistent IDs — skip this cell if you don't
have that data yet; it can be added later.


In [ ]:
# Simple counting evaluation

GT_COUNT = None   # ← set to the known ground-truth apple count for this clip
                  #   e.g.  GT_COUNT = 42

if GT_COUNT is not None:
    pred = unique_track_count          # choose Strategy B or C
    err  = pred - GT_COUNT
    pct  = err / GT_COUNT * 100

    print(f"Ground truth count : {GT_COUNT}")
    print(f"Predicted count    : {pred}")
    print(f"Error              : {err:+d}  ({pct:+.1f} %)")
    print()
    if abs(pct) <= 10:
        print("✓ Within ±10 % — acceptable for MVP.")
    else:
        print("✗ Error exceeds ±10 % — tuning needed.")
else:
    print("GT_COUNT not set — skipping evaluation.")

---

# Section 8 — Export artefacts

Save a JSON summary and the per-frame log CSV so results can be compared
across runs without re-processing the video.


In [ ]:
# Save per-frame log as CSV
csv_path = OUTPUT_DIR / f"frame_log_{run_ts}.csv"
df.to_csv(csv_path, index=False)
print(f"Per-frame log: {csv_path}")

In [ ]:
# Save run summary as JSON
summary = {
    "run_timestamp"       : run_ts,
    "weights"             : str(weights),
    "video"               : str(video_path),
    "n_frames_processed"  : frame_idx,
    "processing_fps"      : round(frame_idx / elapsed, 1),
    "unique_track_ids"    : unique_track_count,
    "line_crossings_in"   : count_line.in_count,
    "line_crossings_out"  : count_line.out_count,
    "cfg"                 : CFG,
}

json_path = OUTPUT_DIR / f"run_summary_{run_ts}.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Run summary    : {json_path}")
print()
print("── Summary ─────────────────────────────────────────────")
for k, v in summary.items():
    if k != "cfg":
        print(f"  {k:<30s} {v}")


In [ ]:
# Cell 8-c  (Kaggle) Everything in /kaggle/working/ is auto-downloadable.
# (Colab)  Download the output folder manually, or copy to Drive:
#
#   from google.colab import files
#   files.download(str(out_path))
#   files.download(str(csv_path))
#   files.download(str(json_path))
#
# Or copy to mounted Drive:
#   shutil.copy(str(out_path), "/content/drive/MyDrive/fruit_tracking/")

print("Artefacts in:", OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name:<50s}  {size:7.1f} KB")